# Continue Phase 3 from the verified 28-response checkpoint

**First use File → Save a copy in Drive, with outputs included, and work in that copy.** Select an **L4 GPU**. This notebook restores the latest published checkpoint captured at **2026-11-10T11:10:25.384475+00:00**, including all 20 payload files, the original eight T4 outcomes, 20 retained first-L4 outcomes, vectors, prior setup/log and runtime records. It then fills only the **170 missing grid positions**. It must never restart from eight.

Reconnection on 2026-11-11 found another fresh runtime after a long monitoring gap. Unarchived attempts after both earlier checkpoints are unknown. A complete result means **198 retained grid outcomes**, not exactly 198 historical attempts. No retained invalid or truncated response is replaced. The original source, prompts, split, vectors, model revision, sampling and strict parser remain fixed; all vectors were fitted on the original T4.

Read `docs/PHASE3_SECOND_RESUME_PROTOCOL.md` and the original recovery/L4 declarations in the pinned checkout. This notebook needs no API key or payment details and does not buy GPU access. Setup refuses any existing live output directory and any nonidentical existing restoration target. The original checkpoint files, the entire 28-row byte prefix and all prior metadata remain immutable.

Run in order. Keep foreground collection visible. Snapshots are emitted at 28, 38, 48, …, 198 retained rows, plus interruption and verified completion. **Save and independently confirm each snapshot in the Drive notebook or an external export.** A screen update, VM-local file, or assumed autosave is not proof of durable preservation. Do not rerun the collection cell or use this notebook to roll back a newer checkpoint.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, re
from datetime import datetime, timezone

COPIED_TO_DRIVE = False  # Set True only in the saved Drive copy, with outputs included.
CHECKPOINT_COMMIT = "06ff192c6662819b893ebdec931f2bbbf2d70b17"
SCIENTIFIC_COMMIT = "ef90362c287774a9d22ffa05a87f9faa960cad4e"
if COPIED_TO_DRIVE is not True:
    raise RuntimeError("Save a copy in Drive with outputs included, then set COPIED_TO_DRIVE=True in that copy.")
if not re.fullmatch(r"[0-9a-f]{40}", CHECKPOINT_COMMIT):
    raise ValueError("CHECKPOINT_COMMIT must be the published full commit hash.")
PROJECT = Path('/content') / ('hint-faithfulness-phase3-latest28-resume-' + CHECKPOINT_COMMIT[:12])
if PROJECT.is_symlink():
    raise RuntimeError("The dedicated checkout must not be a symlink.")
if not PROJECT.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/felmonon/hint-faithfulness.git',str(PROJECT)],check=True)
    subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',CHECKPOINT_COMMIT],check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
changes = subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
if head != CHECKPOINT_COMMIT or changes:
    raise RuntimeError("Existing checkout differs; preserve it and use a fresh dedicated checkout.")
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
os.environ['CUDA_VISIBLE_DEVICES']='0'
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-pilot.txt'],check=True)
print('Pinned checkpoint checkout:',head)
print('No generation has started.')

In [ ]:
import base64, gzip, io, importlib
import numpy as np
import torch, transformers
from collections import Counter
from src.common import LETTERS, read_jsonl, stable_hash
from src.phase3 import load_records, paired_prompts, target_stratified_split
from src.provenance import capture

def stamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()
def sha_file(path):
    return sha_bytes(Path(path).read_bytes())
def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def unpack_bytes(item):
    require(item['encoding']=='gzip+base64','Unknown snapshot encoding.')
    compressed=base64.b64decode(item['base64'],validate=True)
    require(len(compressed)==item['compressed_bytes'] and sha_bytes(compressed)==item['compressed_sha256'],
            'Compressed snapshot payload differs.')
    raw=gzip.decompress(compressed)
    require(len(raw)==item['bytes'] and sha_bytes(raw)==item['sha256'],'Decoded snapshot payload differs.')
    return raw

EXPECTED_SNAPSHOT_FILES = {
  "artifacts/pilot-20-phase3-checkpoint-0430/checkpoint-provenance.json": {
    "sha256": "f012a8d91fc57206913e76a56416a3583a03f0053a691fcaed8e222095fd20a2",
    "bytes": 940
  },
  "artifacts/pilot-20-phase3-checkpoint-0430/execution.log": {
    "sha256": "5b580df84a568b4de5ad1f601844354302fdfaaf01e351f31d1e417d015373d9",
    "bytes": 90146
  },
  "artifacts/pilot-20-phase3-checkpoint-0430/manifest.json": {
    "sha256": "b6ab6444ac32113f1e00ebf3f1cba43a6344f00ada0b431ff7b4fdc03b74565c",
    "bytes": 2531
  },
  "artifacts/pilot-20-phase3-checkpoint-0430/plan.json": {
    "sha256": "5ecc12c4f4d79161de442582f2bea7b89815a0ba06808eb130b00054de52d1f2",
    "bytes": 5109
  },
  "artifacts/pilot-20-phase3-checkpoint-0430/runtime.json": {
    "sha256": "46c1201b9755f0ac8f317773cfdff9e82ed5c5ae966ee3528ff88232e59d0b64",
    "bytes": 19801
  },
  "artifacts/pilot-20-phase3-checkpoint-0430/steering.jsonl": {
    "sha256": "5bec0702c11667104b6a37a3f15487b19dceb7358f771748e467b346ae63044b",
    "bytes": 259338
  },
  "artifacts/pilot-20-phase3-checkpoint-0430/vector_metadata.json": {
    "sha256": "cf3a7ff327d4ef6eb496a0819c8af2539b1d5ee43636db284cbb5266dc0cba20",
    "bytes": 325
  },
  "artifacts/pilot-20-phase3-checkpoint-0430/vectors.npz": {
    "sha256": "9808b154bd3e584c53ba8fe784d3b5d47c67ec131a9c246fe0ca5dfc828e8c57",
    "bytes": 59220
  },
  "artifacts/pilot-20-phase3-resume-notebook-20261110T103905.686089Z/resume-execution.log": {
    "sha256": "64a72187a9469b5c3d524f27c73b721f73b72c244eda3e81e4031be88db799cc",
    "bytes": 88361
  },
  "artifacts/pilot-20-phase3-resume-notebook-20261110T103905.686089Z/setup.json": {
    "sha256": "64f16e6faa983ba54cca6e09ece9b4fe644f7f8e896f6b0f362fa69ebc0f7f35",
    "bytes": 23173
  },
  "artifacts/pilot-20-phase3-v1/execution.log": {
    "sha256": "5b580df84a568b4de5ad1f601844354302fdfaaf01e351f31d1e417d015373d9",
    "bytes": 90146
  },
  "artifacts/pilot-20-phase3-v1/manifest.json": {
    "sha256": "b6ab6444ac32113f1e00ebf3f1cba43a6344f00ada0b431ff7b4fdc03b74565c",
    "bytes": 2531
  },
  "artifacts/pilot-20-phase3-v1/plan.json": {
    "sha256": "5ecc12c4f4d79161de442582f2bea7b89815a0ba06808eb130b00054de52d1f2",
    "bytes": 5109
  },
  "artifacts/pilot-20-phase3-v1/runtime-resume-2026-11-10T10-39-08.868091+00-00.json": {
    "sha256": "b8315b7d73f57a1f3129959a5a349e9981453de6f46e6096c95b6d823025357b",
    "bytes": 19802
  },
  "artifacts/pilot-20-phase3-v1/runtime.json": {
    "sha256": "46c1201b9755f0ac8f317773cfdff9e82ed5c5ae966ee3528ff88232e59d0b64",
    "bytes": 19801
  },
  "artifacts/pilot-20-phase3-v1/steering.jsonl": {
    "sha256": "54f5e311d8e6ceed08629f4bb39f4f0de3b5e7b6a337f74cef2e7812fed73507",
    "bytes": 885213
  },
  "artifacts/pilot-20-phase3-v1/vector_metadata.json": {
    "sha256": "cf3a7ff327d4ef6eb496a0819c8af2539b1d5ee43636db284cbb5266dc0cba20",
    "bytes": 325
  },
  "artifacts/pilot-20-phase3-v1/vectors.npz": {
    "sha256": "9808b154bd3e584c53ba8fe784d3b5d47c67ec131a9c246fe0ca5dfc828e8c57",
    "bytes": 59220
  },
  "docs/PHASE3_L4_RESUME_ADDENDUM.md": {
    "sha256": "88d762777c692619758bba590220d7da8c8bbd15804ded14581c3e68955299fb",
    "bytes": 3448
  },
  "docs/phase3-l4-resume-addendum.json": {
    "sha256": "b9c9afeeddc099297bfd3eb1c9652590afea97ffcf4be03a4c2ad16ef9716607",
    "bytes": 1126
  }
}


LATEST_PUBLICATION_HASHES = {
    'snapshot.json': '92adcb34894143f73f63f74965862d1c895dd7041b2f81e568fbb86c1991b81a',
    'steering.jsonl': '54f5e311d8e6ceed08629f4bb39f4f0de3b5e7b6a337f74cef2e7812fed73507',
    'acquisition-provenance.json': 'f7aaab4e4a3290f511034a87f115dacf61135ed4058bf7d3812d55dea22ba57f',
    'README.md': 'a5f96ed58eeb79b5c27c490ed796eb44eb9785012b0fe3a792c5b49ac24e12bd',
    'checksums.sha256': '3bdc3d73beca7d00d99886018e4da6a85578bbaccb70cde0ce19cc0ece349005',
}
SECOND_PROTOCOL_SHA256 = 'c90533a6be1e8fd563075f9b1d982d03111ff8f380f4ce3b69898965b5e2147d'
LATEST_CHECKPOINT = PROJECT/'artifacts/pilot-20-phase3-checkpoint-1110'
CHECKPOINT = PROJECT/'artifacts/pilot-20-phase3-checkpoint-0430'
PHASE3 = PROJECT/'artifacts/pilot-20-phase3-v1'
SECOND_PROTOCOL_PATH = PROJECT/'docs/PHASE3_SECOND_RESUME_PROTOCOL.md'
ADDENDUM_PATH = PROJECT/'docs/phase3-l4-resume-addendum.json'
ADDENDUM_SHA256 = 'b9c9afeeddc099297bfd3eb1c9652590afea97ffcf4be03a4c2ad16ef9716607'
RESTORED_COUNT = 28
PLANNED_MISSING = 170

# Every publication hash must be fixed before this notebook is executable.
require(all(re.fullmatch(r'[0-9a-f]{64}', value) for value in [*LATEST_PUBLICATION_HASHES.values(), SECOND_PROTOCOL_SHA256]),
        'Publication hashes are not yet frozen. Use the finalized published notebook.')
require(torch.cuda.is_available(), 'A CUDA L4 is required; no checkpoint has been restored.')
gpu = torch.cuda.get_device_properties(0)
require('L4' in gpu.name.split() and gpu.total_memory >= 22*1024**3, 'This continuation requires an L4 with at least 22 GiB.')
EXPECTED_IMPORTED_VERSIONS = {'numpy':'2.2.6','transformers':'5.0.0rc3','torch':'2.14.1+cu130',
    'tokenizers':'0.22.2','huggingface_hub':'1.3.5','datasets':'4.5.0','sklearn':'1.8.0','scipy':'1.16.3','matplotlib':'3.10.8'}
IMPORTED_VERSIONS = {name: importlib.import_module(name).__version__ for name in EXPECTED_IMPORTED_VERSIONS}
require(IMPORTED_VERSIONS == EXPECTED_IMPORTED_VERSIONS,
        'Imported packages differ from the pinned prior runtime. Resolve or restart the kernel before restoring outputs.')
require(not PHASE3.exists(), 'The live output directory already exists. Preserve it; never restore over it.')
for name, expected_hash in LATEST_PUBLICATION_HASHES.items():
    path = LATEST_CHECKPOINT/name
    require(path.is_file() and not path.is_symlink() and sha_file(path)==expected_hash, 'Latest publication differs: '+name)
require(sha_file(SECOND_PROTOCOL_PATH)==SECOND_PROTOCOL_SHA256, 'Second-resume protocol differs.')
require(sha_file(ADDENDUM_PATH)==ADDENDUM_SHA256, 'Frozen L4 declaration differs.')
ADDENDUM = json.loads(ADDENDUM_PATH.read_text())
require(ADDENDUM['allowed_resume_gpu']=='L4', 'Unexpected hardware declaration.')

snapshot_payload = json.loads((LATEST_CHECKPOINT/'snapshot.json').read_text())
require(snapshot_payload['schema']=='phase3-recovery-snapshot-v1'
        and snapshot_payload['captured_at_utc']=='2026-11-10T11:10:25.384475+00:00'
        and snapshot_payload['checkpoint_commit']=='35fc053f65646030b780030fb592861113430ca6'
        and snapshot_payload['scientific_commit']==SCIENTIFIC_COMMIT
        and snapshot_payload['retained_records']==28 and snapshot_payload['original_retained_records']==8
        and snapshot_payload['complete_verified'] is False, 'Unexpected 28-record snapshot identity.')

def decode_checkpoint(payload):
    require(set(payload['files'])==set(EXPECTED_SNAPSHOT_FILES), 'Snapshot must contain exactly the 20 published named files.')
    decoded = {}
    for relative, expected in EXPECTED_SNAPSHOT_FILES.items():
        rel = Path(relative)
        require(not rel.is_absolute() and '..' not in rel.parts, 'Unsafe snapshot path.')
        item = payload['files'][relative]
        require(item['sha256']==expected['sha256'] and item['bytes']==expected['bytes'], 'Published payload metadata differs: '+relative)
        raw = unpack_bytes(item)
        require(len(raw)==expected['bytes'] and sha_bytes(raw)==expected['sha256'], 'Published payload bytes differ: '+relative)
        decoded[relative] = raw
    return decoded

def restoration_targets(decoded):
    # Validate every destination before writing any file; identical tracked files are preserved.
    targets = []
    for relative, raw in decoded.items():
        require(relative in EXPECTED_SNAPSHOT_FILES, 'Restoration path is outside the exact allowlist.')
        path = PROJECT/relative
        require(path.resolve().is_relative_to(PROJECT.resolve())
                and all(not parent.is_symlink() for parent in [path,*path.parents] if parent.is_relative_to(PROJECT)),
                'Symlink or escaped restoration path.')
        if path.exists():
            require(path.is_file() and path.read_bytes()==raw, 'Existing destination differs; nothing may be overwritten: '+relative)
        targets.append((path,raw))
    return targets

decoded = decode_checkpoint(snapshot_payload)
targets_to_restore = restoration_targets(decoded)
old_prefix = str(CHECKPOINT.relative_to(PROJECT))+'/'
run_prefix = str(PHASE3.relative_to(PROJECT))+'/'
EXPECTED_PAYLOAD_HASHES = {name: EXPECTED_SNAPSHOT_FILES[old_prefix+name]['sha256'] for name in
    ('manifest.json','plan.json','runtime.json','steering.jsonl','vector_metadata.json','vectors.npz','execution.log')}
EXPECTED_CODE_HASHES = {
 'src/phase3.py':'57bb529ac9096f8892d3da6253d43b728d7fb562b1c1565a64c5b0691a28fcdc',
 'src/local_model.py':'b7a549ace225611660eb0c27ca03ba12cb1d4a180407b7138c1fd223fdb68766',
 'src/common.py':'665d1d7866e87dac4afd0cff9e12c7be190a57e6daaa9cb692441e1475cc9d9d',
 'scripts/run_phase3_pilot.py':'8e616c5b20f21c01682e215ea2ba0e8b5bf0ed8d145fd529c6497190474ef714'}
for name, expected_hash in EXPECTED_CODE_HASHES.items():
    require(sha_file(PROJECT/name)==expected_hash, 'Frozen scientific source differs: '+name)
for name, expected_hash in EXPECTED_PAYLOAD_HASHES.items():
    require(sha_bytes(decoded[old_prefix+name])==expected_hash, 'Original checkpoint payload differs: '+name)
    if name!='steering.jsonl':
        require(decoded[run_prefix+name]==decoded[old_prefix+name], 'Original immutable payload changed: '+name)
provenance = json.loads(decoded[old_prefix+'checkpoint-provenance.json'])
require(provenance['files_sha256']==EXPECTED_PAYLOAD_HASHES, 'Original checkpoint acquisition hashes differ.')
PLAN = json.loads(decoded[run_prefix+'plan.json'])
require(PLAN==json.loads((PROJECT/'docs/phase3-pilot-plan.json').read_text()), 'Frozen plan differs.')
source_path = PROJECT/PLAN['source_path']
require(sha_file(source_path)==PLAN['source_sha256'], 'Frozen Phase 1 source differs.')
source, identity = load_records(source_path, include_invalid_source_outcomes=True)
baselines, hinted, qids = paired_prompts(source, require_complete=True)
targets = {q: hinted[q][0]['hint_letter'] for q in qids}
train, test = target_stratified_split(qids, targets, PLAN['seed'])
require(train==PLAN['train_question_ids'] and test==PLAN['test_question_ids'], 'Frozen split differs.')
MANIFEST = json.loads(decoded[run_prefix+'manifest.json'])
expected_manifest = {**identity,'source_digest':stable_hash(source),'layer':PLAN['layer'],
 'position':'post_decoder_block_residual_before_final_norm','seed':PLAN['seed'],
 'strengths':[float(s) for s in PLAN['strengths']],'max_new_tokens':PLAN['max_new_tokens'],
 'temperature':PLAN['temperature'],'top_p':PLAN['top_p'],'top_k':PLAN['top_k'],
 'source_dtypes':sorted({r.get('dtype','unknown') for r in source}),'intervention_dtype':PLAN['dtype'],
 'source_prompt_tokens_verified':True,'activation_token_position':'last_original_chat_prompt_token',
 'split_rule':PLAN['split_rule'],'assigned_targets':targets,'training_question_ids':train,'test_question_ids':test}
require(MANIFEST==expected_manifest, 'Manifest differs from frozen source/plan.')
meta = json.loads(decoded[run_prefix+'vector_metadata.json'])
require(meta['training_questions_per_letter']==dict(Counter(targets[q] for q in train))==PLAN['train_target_counts'], 'Training counts differ.')
rng = np.random.default_rng(PLAN['seed'])
with np.load(io.BytesIO(decoded[run_prefix+'vectors.npz']), allow_pickle=False) as vectors:
    require(set(vectors.files)=={f'{d}_{l}' for d in ('vector','random') for l in LETTERS}, 'Vector keys differ.')
    for letter in LETTERS:
        v, r = vectors[f'vector_{letter}'], vectors[f'random_{letter}']
        require(v.shape==r.shape==(2048,) and v.dtype==r.dtype==np.dtype('float32'), 'Vector shape/type differs.')
        require(np.isfinite(v).all() and np.isfinite(r).all(), 'Nonfinite vector.')
        norm = float(np.linalg.norm(v))
        require(norm>0 and np.isclose(norm,meta['vector_norms'][letter],rtol=2e-7,atol=0), 'Vector norm differs.')
        z = rng.standard_normal(v.shape).astype('float32')
        require(np.allclose(r,z*norm/np.linalg.norm(z),rtol=2e-7,atol=1e-7), 'Seeded random control differs.')
ORIGINAL_PREFIX = decoded[old_prefix+'steering.jsonl']
RESTORED_PREFIX = decoded[run_prefix+'steering.jsonl']
require(RESTORED_PREFIX==(LATEST_CHECKPOINT/'steering.jsonl').read_bytes(), 'Published 28-row sidecar differs.')
require(ORIGINAL_PREFIX.endswith(b'\n') and RESTORED_PREFIX.endswith(b'\n') and RESTORED_PREFIX.startswith(ORIGINAL_PREFIX),
        'Both checkpoint prefixes must be complete and preserve the exact original eight rows.')
conditions=[('baseline',None,0.0)]+[(d,l,float(s)) for d in ('vector','random') for l in LETTERS for s in PLAN['strengths'] if s>0]
EXPECTED_IDS=[stable_hash([stable_hash(MANIFEST),q,d,l,s]) for q in test for d,l,s in conditions]
original_rows=[json.loads(line) for line in ORIGINAL_PREFIX.splitlines()]
restored_rows=[json.loads(line) for line in RESTORED_PREFIX.splitlines()]
require(len(EXPECTED_IDS)==len(set(EXPECTED_IDS))==198, 'Unexpected condition grid.')
require(len(original_rows)==8 and [r['record_id'] for r in original_rows]==EXPECTED_IDS[:8], 'Original eight-row grid prefix differs.')
require(len(restored_rows)==28 and [r['record_id'] for r in restored_rows]==EXPECTED_IDS[:28], 'Latest checkpoint is not the 28-row grid prefix.')
OLD_RUNTIME_NAMES = {Path(name).name for name in decoded if name.startswith(run_prefix+'runtime-resume-')}
require(len(OLD_RUNTIME_NAMES)==1, 'Expected the recorded first L4 runtime.')
old_resume = json.loads(decoded[run_prefix+next(iter(OLD_RUNTIME_NAMES))])
require('L4' in old_resume['gpu_name'].split(), 'Prior continuation was not recorded on L4.')

# All 20 files, destinations, source identities and prefixes have passed before restoration starts.
for path, raw in targets_to_restore:
    if path.exists():
        require(path.read_bytes()==raw, 'Destination changed before restoration.')
        continue
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('xb') as output:
        output.write(raw)
METADATA = PROJECT/'artifacts'/('pilot-20-phase3-resume-notebook-'+stamp())
METADATA.mkdir(parents=True, exist_ok=False)
RESUME_LOG = METADATA/'resume-execution.log'
with RESUME_LOG.open('xb'):
    pass
SETUP_PATH = METADATA/'setup.json'
new_runtime = capture()
runtime_changes = {key:{'prior_l4':old_resume.get(key),'current_l4':new_runtime.get(key)}
    for key in ('python','torch_version','cuda_version','gpu_name','nvidia_driver') if old_resume.get(key)!=new_runtime.get(key)}
with SETUP_PATH.open('x', encoding='utf-8') as output:
    json.dump({'checkpoint_commit':CHECKPOINT_COMMIT,'scientific_commit':SCIENTIFIC_COMMIT,
        'hardware_addendum_path':str(ADDENDUM_PATH.relative_to(PROJECT)), 'hardware_addendum_sha256':ADDENDUM_SHA256,'hardware_addendum':ADDENDUM,
        'second_resume_protocol_path':str(SECOND_PROTOCOL_PATH.relative_to(PROJECT)), 'second_resume_protocol_sha256':SECOND_PROTOCOL_SHA256,
        'restored_checkpoint_path':str(LATEST_CHECKPOINT.relative_to(PROJECT)), 'restored_publication_sha256':LATEST_PUBLICATION_HASHES,
        'restored_payload_sha256':{name:item['sha256'] for name,item in EXPECTED_SNAPSHOT_FILES.items()},
        'runtime':new_runtime,'imported_numpy':np.__version__,'imported_transformers':transformers.__version__,
        'imported_versions':IMPORTED_VERSIONS,'runtime_changes':runtime_changes,
        'retained_original_records':8,'restored_retained_records':28,'planned_missing_positions':170,
        'historical_total_attempts':'unknown; unarchived attempts may exist from both earlier sessions'},output,indent=2)
    output.write('\n')
print('Restored all 20 named payloads and 28 responses; prior logs/runtimes and original T4 vectors preserved.')
print('GPU:',gpu.name,'| imported versions:',IMPORTED_VERSIONS,'| runtime differences:',runtime_changes)
print('Only 170 missing positions may be generated. No generation has started.')

## Complete checkpoints and foreground continuation

The helper cell defines a recoverable snapshot. The following cell performs the **170-position continuation** with the unchanged `scripts/run_phase3_pilot.py --resume`. Logs stream to a new, separate `resume-execution.log`; the old setup/log and first resumed runtime remain byte-identical.

Snapshots contain every original 20 payload, the fixed28-row acquisition/publication files, all old and new notebook setup/log directories and runtime-resume files, and the original/L4/second-resume protocol documents. Only the live `steering.jsonl` grows; it must always preserve both the original eight-row and entire 28-row byte prefixes. Its snapshot is cut only at complete JSONL boundaries. Each binary/text payload has original/compressed sizes and SHA256 values and gzip+base64 encoding.

Updates occur at 28, 38, 48,…,198 rows, plus interruption and verified completion. **Save each full structured output to Drive and verify that it persisted.** A local archive or visible display is not evidence of an external backup. Keep the live foreground cell visible and preserve each acquired checkpoint outside the temporary VM.

In [ ]:
import base64, gzip
from IPython.display import JSON, display

FINAL_FILES={'summary.jsonl','explanation_labels.csv','EXPLANATION_SUMMARY.md','LABELING.md','steering_rates.png','completion.json','checksums.sha256'}
SNAPSHOT_DISPLAY=None
LATEST_SNAPSHOT=None
RUN_STARTED=False
RUN_COMPLETED=False

def completed_jsonl(raw):
    boundary=raw.rfind(b'\n')+1
    prefix=raw[:boundary]
    rows=[json.loads(line) for line in prefix.splitlines()]
    return prefix,rows

def current_prefix(max_records=None):
    raw=(PHASE3/'steering.jsonl').read_bytes()
    prefix,rows=completed_jsonl(raw)
    require(prefix.startswith(ORIGINAL_PREFIX) and prefix.startswith(RESTORED_PREFIX),'Original eight-row or entire 28-row byte prefix changed.')
    require(RESTORED_COUNT<=len(rows)<=198 and [r['record_id'] for r in rows]==EXPECTED_IDS[:len(rows)],
            'Duplicate, missing, reordered or unexpected grid IDs.')
    if max_records is not None:
        require(RESTORED_COUNT<=max_records<=len(rows),'Requested snapshot prefix is unavailable.')
        prefix=b''.join(prefix.splitlines(keepends=True)[:max_records]);rows=rows[:max_records]
    return prefix,rows

def pack_bytes(raw):
    compressed=gzip.compress(raw,mtime=0)
    return {'encoding':'gzip+base64','bytes':len(raw),'sha256':sha_bytes(raw),
            'compressed_bytes':len(compressed),'compressed_sha256':sha_bytes(compressed),
            'base64':base64.b64encode(compressed).decode('ascii')}

def unpack_bytes(item):
    require(item['encoding']=='gzip+base64','Unknown snapshot encoding.')
    compressed=base64.b64decode(item['base64'],validate=True)
    require(len(compressed)==item['compressed_bytes'] and sha_bytes(compressed)==item['compressed_sha256'],
            'Compressed snapshot payload differs.')
    raw=gzip.decompress(compressed)
    require(len(raw)==item['bytes'] and sha_bytes(raw)==item['sha256'],'Decoded snapshot payload differs.')
    return raw

def snapshot(complete=False,max_records=None):
    global SNAPSHOT_DISPLAY,LATEST_SNAPSHOT
    prefix,rows=current_prefix(max_records)
    for name,expected in EXPECTED_PAYLOAD_HASHES.items():
        if name!='steering.jsonl':
            require(sha_file(PHASE3/name)==expected,'Restored immutable payload changed: '+name)
    # All old snapshot files stay immutable except the live response prefix.
    for relative,item in EXPECTED_SNAPSHOT_FILES.items():
        if relative!=str((PHASE3/'steering.jsonl').relative_to(PROJECT)):
            require(sha_file(PROJECT/relative)==item['sha256'], 'Prior evidence changed: '+relative)
    for name,expected in LATEST_PUBLICATION_HASHES.items():
        require(sha_file(LATEST_CHECKPOINT/name)==expected,'Published 28-row checkpoint changed: '+name)
    require(sha_file(SECOND_PROTOCOL_PATH)==SECOND_PROTOCOL_SHA256,'Second-resume protocol changed.')
    paths=[PROJECT/name for name in EXPECTED_SNAPSHOT_FILES if name!=str((PHASE3/'steering.jsonl').relative_to(PROJECT))]
    paths += [LATEST_CHECKPOINT/name for name in LATEST_PUBLICATION_HASHES]
    paths += sorted(PHASE3.glob('runtime-resume-*.json'))
    paths += [SECOND_PROTOCOL_PATH, PROJECT/'docs/PHASE3_RESUME_PROTOCOL.md', PROJECT/'docs/PHASE3_L4_ADDENDUM_ERRATUM.md']
    metadata_dirs=sorted((PROJECT/'artifacts').glob('pilot-20-phase3-resume-notebook-*'))
    require(METADATA in metadata_dirs, 'Current notebook metadata directory missing.')
    for directory in metadata_dirs:
        require(directory.is_dir() and not directory.is_symlink()
                and re.fullmatch(r'pilot-20-phase3-resume-notebook-[0-9]{8}T[0-9]{6}\.[0-9]{6}Z', directory.name),
                'Unexpected notebook metadata directory.')
        for name in ('setup.json','resume-execution.log'):
            path=directory/name
            require(path.is_file() and not path.is_symlink(), 'Notebook setup/log missing: '+str(path))
            paths.append(path)
    if complete:
        require(len(rows)==198,'Cannot mark an incomplete snapshot complete.')
        paths += [PHASE3/name for name in sorted(FINAL_FILES) if (PHASE3/name).exists()]
    data={str((PHASE3/'steering.jsonl').relative_to(PROJECT)):pack_bytes(prefix)}
    for path in sorted(set(paths)):
        require(path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(PROJECT.resolve()),
                'Unexpected snapshot file.')
        data[str(path.relative_to(PROJECT))]=pack_bytes(path.read_bytes())
    if complete:
        inventory=[line.split('  ',1)[1] for line in (PHASE3/'checksums.sha256').read_text().splitlines()]
        require(all(str((PHASE3/name).relative_to(PROJECT)) in data for name in inventory),
                'A checksummed artifact is missing from the complete snapshot.')
    payload={'schema':'phase3-recovery-snapshot-v1','captured_at_utc':datetime.now(timezone.utc).isoformat(),
        'checkpoint_commit':CHECKPOINT_COMMIT,'scientific_commit':SCIENTIFIC_COMMIT,
        'retained_records':len(rows),'original_retained_records':8,'restored_checkpoint_records':28,'complete_verified':complete,
        'restored_snapshot_sha256':LATEST_PUBLICATION_HASHES['snapshot.json'],'second_resume_protocol_sha256':SECOND_PROTOCOL_SHA256,
        'historical_total_attempts':'unknown; unarchived attempts may exist from both earlier sessions',
        'durability':'Unverified until this complete output or exported file is saved outside the runtime.',
        'files':data}
    LATEST_SNAPSHOT=payload
    if SNAPSHOT_DISPLAY is None:
        SNAPSHOT_DISPLAY=display(JSON(payload,expanded=False),display_id=True)
    else:
        SNAPSHOT_DISPLAY.update(JSON(payload,expanded=False))
    print(f'Checkpoint output updated: {len(rows)}/198 retained records. Save and verify the Drive copy.',flush=True)
    return payload

def verify_completion():
    prefix,rows=current_prefix()
    require((PHASE3/'steering.jsonl').read_bytes()==prefix,'Trailing incomplete JSONL bytes remain.')
    require(len(rows)==198,'The 198-position grid is incomplete.')
    completion=json.loads((PHASE3/'completion.json').read_text())
    require(completion.get('status')=='complete_fixed_sweep' and completion.get('records')==198,'Completion marker differs.')
    for name,expected in EXPECTED_PAYLOAD_HASHES.items():
        if name!='steering.jsonl':require(sha_file(PHASE3/name)==expected,'Immutable payload changed: '+name)
    names=set()
    for line in (PHASE3/'checksums.sha256').read_text().splitlines():
        expected,name=line.split('  ',1);path=PHASE3/name
        require(name not in names and name!='checksums.sha256','Duplicate or self-referential checksum.')
        require(not path.is_symlink() and path.resolve().is_relative_to(PHASE3.resolve()),'Unsafe checksum path.')
        require(sha_file(path)==expected,'Completed artifact checksum differs: '+name)
        names.add(name)
    require(set(EXPECTED_PAYLOAD_HASHES)|{'completion.json','summary.jsonl','explanation_labels.csv','EXPLANATION_SUMMARY.md','LABELING.md','steering_rates.png'}<=names,
            'Incomplete artifact checksum inventory.')
    require(set(path.name for path in PHASE3.glob('runtime-resume-*.json'))>OLD_RUNTIME_NAMES,'A new resumed-runtime record is required; preserve the old one.')
    from scripts.analyze_phase3_pilot import validate_run
    validate_run(rows,MANIFEST,PLAN,source)
    return len(rows)

print('Snapshot helpers ready. No generation started.')

In [ ]:
import queue, signal, threading, time
require(RUN_STARTED is False,'This collection cell already started; preserve its output instead of rerunning it.')
require(COPIED_TO_DRIVE is True,'Use a saved Drive copy with outputs included.')
require(RESUME_LOG.is_file() and RESUME_LOG.stat().st_size==0,'New resume log is not empty; preserve the prior run.')
require(len(current_prefix()[1])==RESTORED_COUNT,'This notebook starts only from the verified 28-row checkpoint; never restart from 8.')
RUN_STARTED=True
command=[sys.executable,'-u','scripts/run_phase3_pilot.py','--resume']
events=queue.Queue();reader_errors=[]
process=None;reader=None
last_snapshot_n=RESTORED_COUNT
snapshot()  # Recoverable initial state, still requiring an external save.
log=RESUME_LOG.open('ab')
try:
    process=subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             start_new_session=True,bufsize=0)
    def copy_output():
        try:
            for raw in iter(process.stdout.readline,b''):
                log.write(raw);log.flush()
                text=raw.decode('utf-8',errors='replace')
                if 'Saved steering response ' in text or 'PHASE3_' in text:
                    events.put(text.rstrip())
        except BaseException as exc:
            reader_errors.append(exc)
    reader=threading.Thread(target=copy_output,daemon=True);reader.start()
    while process.poll() is None or reader.is_alive():
        if process.poll() is not None and process.returncode!=0:
            raise RuntimeError('Runner exited unsuccessfully; stopping any surviving child and preserving a snapshot.')
        while not events.empty():print(events.get_nowait(),flush=True)
        if reader_errors:raise RuntimeError('Log reader failed; preserve partial evidence.') from reader_errors[0]
        count=len(current_prefix()[1])
        while count-last_snapshot_n>=10:
            last_snapshot_n+=10;snapshot(max_records=last_snapshot_n)
        time.sleep(.5)
    reader.join()
    while not events.empty():print(events.get_nowait(),flush=True)
    if reader_errors:raise RuntimeError('Log reader failed.') from reader_errors[0]
    require(process.wait()==0,'Runner stopped. Preserve the latest snapshot and resume log; do not rerun retained outputs.')
    log.flush()
    verify_completion()
    RUN_COMPLETED=True
    snapshot(complete=True)
    print('Verified 198 retained grid outcomes. Historical total attempts remain unknown. Run the export cell.')
except BaseException:
    try:
        if process is not None:
            # Also stop descendants if their leader has already exited.
            try:os.killpg(process.pid,signal.SIGTERM)
            except ProcessLookupError:pass
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError:pass
                process.wait(timeout=10)
        if reader is not None:
            reader.join(timeout=10)
            if reader.is_alive() and process is not None:
                try:os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError:pass
                reader.join(timeout=10)
    except Exception as cleanup_error:
        print('Process cleanup warning:',type(cleanup_error).__name__)
    finally:
        try:log.flush()
        except Exception as flush_error:print('Log flush warning:',type(flush_error).__name__)
        try:snapshot(complete=False)
        except Exception as backup_error:print('Final snapshot failed:',type(backup_error).__name__,'Preserve the prior saved snapshot.')
    raise
finally:
    log.close()

In [ ]:
import zipfile
from google.colab import files
require(RUN_COMPLETED is True,'The foreground run has not completed; preserve the partial JSON snapshot instead.')
verify_completion()
payload=snapshot(complete=True)
export_stamp=stamp()
archive_path=Path('/content')/f'pilot-20-phase3-resumed-from28-{export_stamp}.zip'
snapshot_path=Path('/content')/f'pilot-20-phase3-snapshot-{export_stamp}.json'
with snapshot_path.open('x',encoding='utf-8') as stream:
    json.dump(payload,stream,separators=(',',':'));stream.write('\n')
with zipfile.ZipFile(archive_path,'x',zipfile.ZIP_DEFLATED) as archive:
    for relative,item in sorted(payload['files'].items()):
        rel=Path(relative)
        require(not rel.is_absolute() and '..' not in rel.parts,'Unsafe export path.')
        archive.writestr(relative,unpack_bytes(item))
    archive.writestr('export-manifest.json',json.dumps({
        'schema':payload['schema'],'checkpoint_commit':CHECKPOINT_COMMIT,'retained_records':198,'restored_checkpoint_records':28,
        'historical_total_attempts':payload['historical_total_attempts'],
        'files':{name:{'bytes':item['bytes'],'sha256':item['sha256']} for name,item in payload['files'].items()}},indent=2)+'\n')
print('ZIP:',archive_path.name,'bytes:',archive_path.stat().st_size,'SHA256:',sha_file(archive_path))
print('Fallback JSON:',snapshot_path.name,'bytes:',snapshot_path.stat().st_size,'SHA256:',sha_file(snapshot_path))
print('The complete structured JSON snapshot is also in the foreground cell output. Save and verify the Drive copy.')
try:
    files.download(str(archive_path))
except Exception as error:
    print('Browser download did not complete:',type(error).__name__)
    print('Use the saved full JSON output or download the standalone JSON from the Files panel.')

## If another interruption occurs

Preserve the newest complete saved JSON snapshot and its acquisition history. This notebook restores exactly 28 rows; **do not use it to roll back a later prefix**. Publish and verify a later checkpoint, document the interruption, and use a deliberate continuation that binds that newer entire byte prefix. Never replace retained failures, silently refit vectors, select outcomes by quality, or claim that unarchived attempts did not happen.

For every snapshot file, decode base64, verify compressed size/hash, gzip-decompress, then verify original size/hash before any restoration. Reject missing/extra files, unexpected paths, symlinks and nonidentical existing destinations. A truncated display or JSON object is not a recoverable checkpoint. The structured snapshot contains only named experiment evidence, not credentials or model weights.

After a verified 198-record export, run the full artifact audit, primary paired/attrition analysis and predeclared five-question hardware sensitivity. The final report must disclose 8 T4 + 20 first-L4 + 170 second-L4 retained outcomes, actual runtime changes and unknown historical attempts from both earlier sessions. This notebook supplies no explanation labels, human validation, steering-success or faithfulness claim.